# **Symmetry Reduction in the Graph Coloring Problem: ILP Formulation**

This notebook presents an Integer Linear Programming formulation for the graph coloring problem and investigates the role of color permutation symmetry. We demonstrate how relabeling colors generates equivalent optimal colorings and how first-use ordering constraints can eliminate redundant solutions while preserving optimality.

---

## *Mathematical Background*

### **Graph Coloring as Integer Linear Program**

Given an undirected graph $G = (V, E)$, the graph coloring problem seeks an assignment of colors to vertices such that no two adjacent vertices share the same color, while minimizing the total number of colors used.<sup>[[1]](#ref1)

Let $k = |V|$ denote the number of available colors, which provides a trivial upper bound since we can always assign a distinct color to each vertex. We introduce two families of binary decision variables: $x_{vc} \in \{0,1\}$ for each vertex $v \in V$ and color $c \in \{1, \ldots, k\}$, where $x_{vc} = 1$ indicates that vertex $v$ is assigned color $c$; and $y_c \in \{0,1\}$ for each color $c \in \{1, \ldots, k\}$, where $y_c = 1$ indicates that color $c$ is used in the solution.

The objective function minimizes the number of colors used:

\begin{equation*}
\min \sum_{c=1}^{k} y_c
\end{equation*}

We impose three families of constraints. First, each vertex must receive exactly one color:

\begin{equation*}
\sum_{c=1}^{k} x_{vc} = 1, \quad \forall v \in V
\end{equation*}

Second, adjacent vertices must receive different colors:

\begin{equation*}
x_{uc} + x_{vc} \leq 1, \quad \forall \{u,v\} \in E, \quad \forall c \in \{1, \ldots, k\}
\end{equation*}

Third, a color is active if and only if it is assigned to at least one vertex:

\begin{equation*}
x_{vc} \leq y_c, \quad \forall v \in V, \quad \forall c \in \{1, \ldots, k\}
\end{equation*}

The optimal value of this ILP equals the chromatic number $\chi(G)$: the minimum number of colors required for a proper coloring of $G$.

---

### **Graph Symmetries**

For the graph coloring problem, two independent sources of symmetry arise.<sup>[[2]](#ref2), [[3]](#ref3)</sup>

The first source is structural: graph automorphisms. A bijection $\pi: V \to V$ is an automorphism of $G$ if $\{u,v\} \in E \iff \{\pi(u), \pi(v)\} \in E$. The set of all automorphisms forms the group $\mathrm{Aut}(G)$ under composition. Given a feasible coloring $\mathbf{x}$, the reassignment $x'_{\pi(v),c} = x_{vc}$ (permuting vertex indices while preserving color assignments) yields another feasible coloring with the same objective value.

The second, and more fundamental, source is **color permutation symmetry**. Let $\sigma \in S_k$ be any permutation of the $k$ color labels. Given a proper coloring $\mathbf{x}$, the relabeled coloring $\mathbf{x}'$ defined by

\begin{equation*}
x'_{v,\sigma(c)} = x_{vc}, \quad \forall v \in V, \quad \forall c \in \{1, \ldots, k\}
\end{equation*}

is also a proper coloring with the same objective value. The symmetric group $S_k$ therefore acts on the solution space, partitioning it into equivalence classes. At optimality, all $\chi(G)$ colors are used, and every optimal coloring belongs to an orbit of size exactly $\chi(G)!$ under this action:

\begin{equation*}
\mathrm{Orb}(\mathbf{x}) = \{\sigma(\mathbf{x}) : \sigma \in S_{\chi(G)}\}
\end{equation*}

Since color symmetry is independent of graph structure, it is always present regardless of $\mathrm{Aut}(G)$. Together, the two groups generate the full symmetry of the problem, and searching over the quotient space $\mathcal{F} / (\mathrm{Aut}(G) \times S_{\chi(G)})$ suffices to identify all essentially distinct optimal colorings.

---

### **Symmetry Breaking**

To eliminate the $\chi(G)!$ equivalent solutions generated by color permutations, we impose a *first-use ordering* constraint.<sup>[[2]](#ref2), [[3]](#ref3)</sup> The idea is to enforce that color labels are introduced in increasing order when scanning vertices by index: color $1$ must be the first new color encountered, color $2$ the second, and so on.

Formally, vertex $v$ may receive color $c \geq 2$ only if some vertex $u < v$ has already received color $c - 1$:

\begin{equation*}
x_{vc} \leq \sum_{u < v} x_{u,c-1}, \quad \forall v \in V, \quad \forall c \in \{2, \ldots, k\}
\end{equation*}

For the first vertex ($v = 1$), the right-hand side is zero for all $c \geq 2$, forcing $x_{1,c} = 0$: vertex $1$ always receives color $1$. More generally, color $c$ can only be introduced after color $c - 1$ has been used, establishing a canonical ordering of color labels across all feasible solutions.

A valid symmetry-breaking scheme must satisfy two properties:<sup>[[3]](#ref3)</sup>
- *Completeness*: for any optimal coloring, there exists a unique relabeling of colors such that the first-use ordering is satisfied. Given any proper coloring, one can always rename colors so that the first new color encountered (scanning vertices in increasing index order) receives label $1$, the second receives label $2$, and so on.
- *Soundness*: the canonical representative has the same objective value as any other element of its orbit, since the number of colors used is invariant under color permutation.

The constraint eliminates all $\chi(G)! - 1$ non-canonical permutations, retaining exactly one representative per color-permutation orbit.

---

### **References**

<a id="ref1"></a>
<sup>[[1]](#ref1)</sup> Garey, M. R., & Johnson, D. S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W. H. Freeman.

<a id="ref2"></a>
<sup>[[2]](#ref2)</sup> Margot, F. (2010). Symmetry in Integer Linear Programming. In *50 Years of Integer Programming 1958–2008* (pp. 647–686). Springer.

<a id="ref3"></a>
<sup>[[3]](#ref3)</sup> Liberti, L. (2012). Symmetry in Mathematical Programming. In *Mixed Integer Nonlinear Programming* (pp. 263–283). Springer.

## *Computational Implementation*

In [1]:
using JuMP
using HiGHS
println("Packages loaded | Julia ", VERSION)

Packages loaded | Julia 1.11.5


In [2]:
function build_gc_model(edges, N, k;
                        symmetry_breaking=false, fixed_obj=nothing)
    """
    Builds an integer linear programming model for the graph coloring problem.

    Args:
        edges: List of edges (tuples (u,v)) of the graph
        N: Total number of vertices in the graph
        k: Number of available colors (upper bound)
        symmetry_breaking: (optional) Enables first-use ordering constraints
        fixed_obj: (optional) Fixes the objective value (for enumerating optimal solutions)

    Returns: (model, x, y) - The JuMP model and decision variables
    """

    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, x[1:N, 1:k], Bin)
    @variable(model, y[1:k], Bin)

    @objective(model, Min, sum(y[c] for c in 1:k))

    # Each vertex receives exactly one color
    for v in 1:N
        @constraint(model, sum(x[v,c] for c in 1:k) == 1)
    end

    # Adjacent vertices must receive different colors
    for (u,v) in edges, c in 1:k
        @constraint(model, x[u,c] + x[v,c] <= 1)
    end

    # Color c is active if at least one vertex uses it
    for v in 1:N, c in 1:k
        @constraint(model, x[v,c] <= y[c])
    end

    # Fix objective value for solution enumeration
    if fixed_obj !== nothing
        @constraint(model, sum(y[c] for c in 1:k) == fixed_obj)
    end

    # Symmetry breaking: first-use ordering
    # x[v,c] <= sum(x[u,c-1] for u < v): color c can only be assigned to vertex v
    # if color c-1 has already been assigned to some vertex u with u < v
    if symmetry_breaking
        for c in 2:k, v in 1:N
            @constraint(model, x[v,c] <= sum(x[u,c-1] for u in 1:(v-1); init=0))
        end
    end

    return model, x, y
end

function extract_coloring(x, N, k)
    """
    Extracts the color assignment from the binary decision variables.

    Args:
        x: Model decision variables (color assignments)
        N: Number of vertices
        k: Number of available colors

    Returns: Vector with the color assigned to each vertex
    """

    coloring = zeros(Int, N)
    for v in 1:N
        for c in 1:k
            if value(x[v,c]) > 0.5
                coloring[v] = c
                break
            end
        end
    end
    return coloring
end

function enumerate_colorings(edges, N, k, opt_val;
                              symmetry_breaking=false)
    """
    Enumerates all optimal colorings (up to 200) using the minimum number of colors.

    Args:
        edges: List of graph edges
        N: Total number of vertices
        k: Number of available colors
        opt_val: Optimal value (minimum number of colors) to match
        symmetry_breaking: (optional) Enables first-use ordering constraints

    Returns: Vector of colorings (each coloring is a Vector{Int})
    """

    solutions = Vector{Vector{Int}}()

    for _ in 1:200
        model, x, y = build_gc_model(edges, N, k,
                                      symmetry_breaking=symmetry_breaking,
                                      fixed_obj=opt_val)

        # Add no-good cuts to exclude previously found colorings:
        # at least one vertex must change its color assignment
        for prev_col in solutions
            @constraint(model, sum(x[v, prev_col[v]] for v in 1:N) <= N-1)
        end

        optimize!(model)
        termination_status(model) != OPTIMAL && break

        coloring = extract_coloring(x, N, k)
        push!(solutions, coloring)
    end

    return solutions
end

enumerate_colorings (generic function with 1 method)

### **Example**: Cycle Graph $C_5$

#### Problem Instance

Consider the cycle graph $G = C_5$ with five vertices arranged in a pentagon:

\begin{equation*}
\begin{array}{ccccc}
 & & [1] & & \\[4pt]
 & \diagup & & \diagdown & \\[4pt]
[5] & & & & [2] \\[4pt]
 & \diagdown & & \diagup & \\[4pt]
 & [4] & - & [3] &
\end{array}
\end{equation*}

The vertex set is $V = \{1, 2, 3, 4, 5\}$ and the edge set is $E = \{\{1,2\}, \{2,3\}, \{3,4\}, \{4,5\}, \{5,1\}\}$. We seek a minimum proper coloring from source $k = |V| = 5$ available colors. For this example, we set $k = 3$ since $C_5$ is known to have chromatic number $\chi(C_5) = 3$ (as confirmed by the ILP); the upper bound $k = |V| = 5$ would additionally involve color-subset symmetry, generating $\binom{5}{3} \cdot 3! = 300$ equivalent optimal colorings without symmetry breaking.

The cycle $C_5$ is 3-chromatic because it is an odd cycle: it cannot be properly 2-colored (a proper 2-coloring of a cycle requires the cycle to have even length), and three colors suffice by assigning colors $1, 2, 1, 2, 3$ along the cycle.

The graph possesses a rich symmetry structure. Its automorphism group is the dihedral group $\mathrm{Aut}(C_5) \cong D_5$, generated by the rotation $\rho: v \mapsto (v \bmod 5) + 1$ and the reflection $\sigma: v \mapsto 6 - v$, with $|\mathrm{Aut}(C_5)| = 10$. In addition, all optimal colorings using exactly $\chi(C_5) = 3$ colors exhibit the full color permutation symmetry of $S_3$, with $|S_3| = 3! = 6$. This notebook addresses the $S_3$ color symmetry; the vertex symmetry from $\mathrm{Aut}(C_5)$ is discussed as future work.

#### Problem Instance

In [3]:
edges_1 = [(1,2), (2,3), (3,4), (4,5), (5,1)]
N_1 = 5
k_1 = 3

println("N = ", N_1, ", |E| = ", length(edges_1))
println("Available colors: k = ", k_1)

N = 5, |E| = 5
Available colors: k = 3


#### Solution Enumeration

In [4]:
# Find optimal value
model_1, x_1, y_1 = build_gc_model(edges_1, N_1, k_1)
optimize!(model_1)
opt_val_1 = round(Int, objective_value(model_1))

# Enumerate solutions
solutions_baseline_1 = enumerate_colorings(edges_1, N_1, k_1, opt_val_1,
                                            symmetry_breaking=false)

solutions_reduced_1  = enumerate_colorings(edges_1, N_1, k_1, opt_val_1,
                                            symmetry_breaking=true)

println("Optimal value = ", opt_val_1)

println("\nBaseline colorings:")
for (i, col) in enumerate(solutions_baseline_1)
    println("C", i, ": ", col)
end

println("\nSymmetry-reduced colorings:")
for (i, col) in enumerate(solutions_reduced_1)
    println("C", i, ": ", col)
end

Optimal value = 3

Baseline colorings:
C1: [1, 2, 1, 2, 3]
C2: [1, 3, 1, 2, 3]
C3: [3, 2, 1, 2, 1]
C4: [3, 2, 3, 2, 1]
C5: [3, 2, 1, 3, 1]
C6: [3, 1, 2, 3, 1]
C7: [2, 3, 2, 3, 1]
C8: [1, 3, 2, 3, 2]
C9: [1, 3, 1, 3, 2]
C10: [1, 3, 2, 1, 3]
C11: [1, 2, 3, 1, 3]
C12: [2, 1, 3, 1, 3]
C13: [2, 1, 3, 2, 1]
C14: [3, 2, 3, 1, 2]
C15: [3, 2, 1, 3, 2]
C16: [3, 1, 2, 3, 2]
C17: [2, 1, 2, 1, 3]
C18: [2, 1, 2, 3, 1]
C19: [2, 3, 1, 2, 1]
C20: [1, 2, 3, 2, 3]
C21: [2, 1, 3, 2, 3]
C22: [3, 1, 3, 2, 1]
C23: [3, 1, 3, 1, 2]
C24: [3, 1, 2, 1, 2]
C25: [2, 3, 2, 1, 3]
C26: [2, 3, 1, 2, 3]
C27: [2, 3, 1, 3, 1]
C28: [1, 2, 3, 1, 2]
C29: [1, 2, 1, 3, 2]
C30: [1, 3, 2, 1, 2]

Symmetry-reduced colorings:
C1: [1, 2, 3, 1, 2]
C2: [1, 2, 3, 1, 3]
C3: [1, 2, 1, 3, 2]
C4: [1, 2, 1, 2, 3]
C5: [1, 2, 3, 2, 3]


#### Results Analysis

The optimal value is $3$, confirming that $\chi(C_5) = 3$: the five-cycle is an odd cycle and therefore cannot be properly 2-colored.

The total number of proper 3-colorings of $C_5$ is given by the chromatic polynomial $P(C_5, k) = (k-1)^5 - (k-1)$, which for $k = 3$ evaluates to $P(C_5, 3) = 2^5 - 2 = 30$. Since $C_5$ has no proper 2-coloring, all 30 colorings use exactly the three available colors. Without symmetry breaking, the enumeration confirms all 30 solutions.

Each optimal coloring uses all three colors, so no coloring is fixed by any non-identity permutation of $S_3$. The 30 solutions therefore partition into exactly $30 / 3! = 5$ orbits of size $6$ each under color permutation. Each orbit contains the six relabelings of a single essentially distinct coloring.

Introducing the first-use ordering constraint, the enumeration yields the 5 canonical representatives:

\begin{equation*}
\begin{aligned}
C_1 &: [1, 2, 1, 2, 3] \\
C_2 &: [1, 2, 1, 3, 2] \\
C_3 &: [1, 2, 3, 1, 2] \\
C_4 &: [1, 2, 3, 1, 3] \\
C_5 &: [1, 2, 3, 2, 3]
\end{aligned}
\end{equation*}

In each canonical coloring, vertex $1$ receives color $1$ and vertex $2$ receives color $2$: the constraint forces color $1$ to be introduced at $v = 1$ and forbids color $3$ from appearing before color $2$ has been seen, which in turn forces vertex $2$ (adjacent to vertex $1$, hence unable to use color $1$) to use color $2$.

The reduction factor is $30 / 5 = 6 = 3!$, matching the order of the color permutation group $S_3$ and confirming that the symmetry was fully broken.

The 5 canonical solutions still exhibit residual symmetry arising from $\mathrm{Aut}(C_5)$. Under the reflection $\sigma: v \mapsto 6 - v$ (which maps $1 \leftrightarrow 5$ and $2 \leftrightarrow 4$, fixing $3$) composed with an appropriate color relabeling:

- $C_1 = [1,2,1,2,3]$ maps to $\sigma(C_1) = [3,2,1,2,1]$, whose canonical form under the relabeling $(1 \leftrightarrow 3)$ is $[1,2,3,2,3] = C_5$.
- $C_2 = [1,2,1,3,2]$ maps to $\sigma(C_2) = [2,3,1,2,1]$, whose canonical form is $[1,2,3,1,3] = C_4$.
- $C_3 = [1,2,3,1,2]$ maps to $\sigma(C_3) = [2,1,3,2,1]$, whose canonical form is $[1,2,3,1,2] = C_3$.

Thus $C_1 \leftrightarrow C_5$ and $C_2 \leftrightarrow C_4$ under $\sigma$ composed with a color relabeling, while $C_3$ is self-symmetric. The 5 canonical colorings therefore form 3 distinct classes under the full symmetry group $\mathrm{Aut}(C_5) \times S_3$: $\{C_1, C_5\}$, $\{C_2, C_4\}$, and $\{C_3\}$. Achieving complete symmetry breaking would require additional vertex-symmetry constraints that are not implemented here.

## **Discussion**

The results obtained in this notebook illustrate the effect of first-use ordering constraints on the graph coloring problem formulated as an Integer Linear Program.

In contrast to the shortest path problem, where symmetry arose exclusively from the structural automorphisms of the graph, graph coloring exhibits an additional and more pervasive source of symmetry: color permutation. The group $S_{\chi(G)}$ acts on every instance regardless of $\mathrm{Aut}(G)$, generating $\chi(G)!$ equivalent solutions per essentially distinct coloring. For the cycle $C_5$ with $\chi(C_5) = 3$, this produced $3! = 6$ redundant solutions per orbit and a total of $30$ equivalent optimal colorings. The first-use ordering constraint reduced this to $5$ canonical representatives, achieving a reduction factor of exactly $3! = 6$ and confirming both completeness and soundness of the scheme.

A distinguishing feature of color symmetry is its factorial growth with the chromatic number: $\chi(G)! = 6$ for $\chi = 3$, $24$ for $\chi = 4$, and $120$ for $\chi = 5$. As $\chi(G)$ increases, the proportion of the search space occupied by redundant solutions grows rapidly, making symmetry breaking increasingly critical for enumeration efficiency.

The example also reveals the layered structure of symmetry in graph coloring. After eliminating color permutation symmetry via first-use ordering, a residual symmetry remains due to the vertex automorphisms in $\mathrm{Aut}(C_5) \cong D_5$. The 5 canonical colorings are not all essentially distinct: under the combined action of $\mathrm{Aut}(C_5)$ and $S_3$, they collapse into 3 equivalence classes. Complete symmetry breaking would require additional constraints that exploit the vertex symmetry, such as those derived from the orbits of $D_5$ acting on vertex-color pairs. The systematic construction of such hierarchical constraints, as well as the extension of these techniques to the other combinatorial optimization problems in this repository, constitutes a direction for future work.